In [0]:
dbutils.widgets.text("file_id", "")
dbutils.widgets.text("adf_run_id", "")

file_id = dbutils.widgets.get("file_id")
adf_run_id = dbutils.widgets.get("adf_run_id")

print("File ID:", file_id)
print("ADF Run ID:", adf_run_id)

In [0]:
control_table = "dbw_healthcare_de_dev.etl.file_control"

df_before = spark.sql(f"""
    SELECT
        pipeline_status,
        processing_stage,
        adf_run_id
    FROM {control_table}
    WHERE file_id = :file_id
""", args={"file_id": file_id})

before_rows = df_before.collect()

if len(before_rows) != 1:
    raise RuntimeError(
        f"Expected exactly one control record for {file_id}"
    )

before = before_rows[0]

if (
    before["pipeline_status"] != "PENDING"
    or before["processing_stage"] != "REGISTERED"
):
    raise RuntimeError(
        f"File is not eligible for a new claim: "
        f"status={before['pipeline_status']}, "
        f"stage={before['processing_stage']}"
    )

print("Pre-claim eligibility verified")

In [0]:
if not file_id.strip() or not adf_run_id.strip():
    raise ValueError("file_id and adf_run_id are required")

claim_rows = spark.sql(
    """
    CALL dbw_healthcare_de_dev.etl.usp_claim_file(
        p_file_id => :file_id,
        p_adf_run_id => :adf_run_id
    )
    """,
    args={
        "file_id": file_id,
        "adf_run_id": adf_run_id
    }
).collect()

if len(claim_rows) != 1:
    raise RuntimeError(
        f"Unexpected claim procedure result: {len(claim_rows)} rows"
    )

claim_success = claim_rows[0]["claim_success"]
rows_claimed = claim_rows[0]["rows_claimed"]

if claim_success is not True or rows_claimed != 1:
    raise RuntimeError(
        f"File claim rejected: file_id={file_id}, "
        f"rows_claimed={rows_claimed}"
    )

print("File claimed successfully")
print("Rows claimed:", rows_claimed)

In [0]:
df_claim = spark.sql("""
    SELECT
        file_id,
        batch_id,
        pipeline_status,
        processing_stage,
        adf_run_id
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
""", args={"file_id": file_id})

rows = df_claim.collect()

if len(rows) != 1:
    raise RuntimeError("File control record not found or duplicated")

row = rows[0]

claim_success = (
    row["pipeline_status"] == "IN_PROGRESS"
    and row["processing_stage"] == "BRONZE"
    and row["adf_run_id"] == adf_run_id
)

if not claim_success:
    raise RuntimeError(
        f"File claim not confirmed for file_id={file_id}"
    )

print("File claim verified successfully")
display(df_claim)

In [0]:
import json

claim_result = {
    "file_id": file_id,
    "adf_run_id": adf_run_id,
    "claim_success": claim_success,
    "rows_claimed": rows_claimed
}

result_json = json.dumps(claim_result)

print(result_json)
dbutils.notebook.exit(result_json)